# 2 · Risk drivers: WoE, IV and chi-square

Every bin edge, rare-category mapping, clipping bound and feature-selection statistic is fitted on the training 60%. Validation and test labels do not enter these calculations. WoE is log(good distribution / bad distribution), with 0.5 additive smoothing; missing values have a separate bucket.

In [ ]:
from pathlib import Path
import os
import sys
import json
import pandas as pd
from IPython.display import display, Image, Markdown

# The executor supplies these paths. Interactive runs default to local exports.
ROOT = Path(os.environ.get('HOME_CREDIT_PROJECT_ROOT', str(Path.cwd()))).resolve()
if not (ROOT / 'src' / 'analysis.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.analysis import ensure_analysis, load_features, eda_tables

INPUT = Path(os.environ.get('HOME_CREDIT_FEATURES', str(ROOT / 'exports' / 'applicant_features.csv')))
OUTPUT = Path(os.environ.get('HOME_CREDIT_ANALYSIS', str(ROOT / 'outputs' / 'analysis')))
STATUS = os.environ.get('HOME_CREDIT_DATA_STATUS', 'real')
LGD = float(os.environ.get('HOME_CREDIT_LGD', '0.45'))
SEED = int(os.environ.get('HOME_CREDIT_SEED', '42'))
manifest = ensure_analysis(INPUT, OUTPUT, data_status=STATUS, lgd=LGD, seed=SEED)
get_ipython().run_line_magic('matplotlib', 'inline')
display(Markdown(f'**Data status: {manifest["data_status"].upper()}**. '
    + ('Synthetic pipeline validation; no Home Credit conclusions.' if STATUS != 'real'
       else 'Computed from the supplied accepted-cohort export.')))


In [ ]:
iv = pd.read_csv(OUTPUT / 'iv_summary.csv')
woe = pd.read_csv(OUTPUT / 'woe_bins.csv')
chi = pd.read_csv(OUTPUT / 'chi_square_tests.csv')
display(iv.head(25))
display(chi.head(25))

## Read a driver

High IV indicates separation of good/bad distributions in the training sample. IV is an association screen, not a guarantee of future performance. Chi-square tests use Benjamini–Hochberg false-discovery correction and report Cramér's V. Sparse expected counts are flagged; significance in a large dataset can coexist with a small practical effect.

In [ ]:
top_feature = iv.iloc[0].feature
display(Markdown(f'### Train-only bins: {top_feature}'))
display(woe[woe.feature == top_feature])
assert set(woe.fit_split.unique()) == {'train'}
assert set(iv.fit_split.unique()) == {'train'}

In [ ]:
display(pd.DataFrame([{'feature': name, 'kind': spec['kind'],
    'clip_low': spec.get('clip_low'), 'clip_high': spec.get('clip_high'),
    'numeric_edges': spec.get('edges'), 'rare_minimum': spec.get('minimum_count')}
    for name, spec in manifest['woe_specs'].items()]).head(30))

## Missing, rare and unseen values

Missing is explicitly included even when absent in train. Rare observed levels share a train-defined bucket. Unseen levels receive neutral WoE 0 when no rare bucket exists; otherwise they use the train rare-level bucket. Outlying future values are clipped using train 1st/99th percentiles before applying the persisted bins. The statistical tests are exploratory because continuous bins are data derived.